# Border 2 — Input Data Preparation

Retrieves and validates all inputs needed by `border2_experiment.ipynb`:

| Step | Data | Source | Output |
|------|------|--------|--------|
| 1 | Fire perimeters | WFIGS (already in repo) | copied to `data/border2_inputs/` |
| 2 | FIRMS hotspots | NASA FIRMS CSV API | `firms_hotspots.csv` |
| 3 | Weather observations | WIFIRE Firemap pylaski | `weather_obs.csv` |
| 4 | Landscape (LCP) | LANDFIRE → lcpmake via farsite-runner | `landscape.lcp` |

All files land in **`data/border2_inputs/`**.  
Re-running is safe — each step skips if the output already exists.

**Setup**: There is no need to re-run these cells if install_packages.sh has already been executed once.

In [1]:
!chmod +x src/TestFARSITE src/lcpmake install_packages.sh

In [2]:
!bash install_packages.sh

Transaction

  Prefix: /opt/conda/envs/gdal_env

  Updating specs:

   - python=3.11
   - gdal
   - geopandas
   - shapely
   - contextily
   - basemap
   - ipykernel


  Package                         Version  Build                 Channel          Size
────────────────────────────────────────────────────────────────────────────────────────
  Install:
────────────────────────────────────────────────────────────────────────────────────────

  + python_abi                       3.11  9_cp311               conda-forge       7kB
  + ca-certificates            2026.10.23  hbd8a1cb_0            conda-forge     130kB
  + tzdata                          2026e  h5d301ff_0            conda-forge     119kB
  + libzlib                         1.3.2  h25fd6f3_3            conda-forge      64kB
  + libgomp                        16.2.0  he0feb66_7            conda-forge     641kB
  + zstd                            1.5.7  hb78ec9c_7            conda-forge     601kB
  + zlib                        

## Imports & Configuration

**Important**: Switch to the Python kernel named `Python (gdal_env)` before proceeding with the rest of the notebook.

In [1]:
import sys
sys.path.insert(0, 'src')

import os
import csv
import time
import json
import shutil
import requests
from pathlib import Path
import warnings
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load credentials from user_info.env
env_file = Path('user_info.env')
if env_file.exists():
    load_dotenv(env_file)
    print('Loaded user_info.env')
else:
    print('[WARN] user_info.env not found — LANDFIRE and FIRMS steps may fail')

from config import DATA_DIR, OUTPUT_DIR
from firemap import fetch_fire_perimeters, _fetch_wfigs, _fetch_nifc
from farsite import download_landfire_data, generate_lcp_from_rasters


# ── Bounding box (Border 2 fire area, with buffer) ───────────────────────
BBOX = (-117.1, 32.4, -116.6, 32.8)   # (minx, miny, maxx, maxy) WGS84
MINX, MINY, MAXX, MAXY = BBOX
FIRE_CENTER_LAT =  32.59588662537945
FIRE_CENTER_LON = -116.85708741551528

# ── Fire dates ───────────────────────────────────────────────────────────
DATE_START = '2025-01-23'
DATE_END   = '2025-01-27'

# ── Output directory ─────────────────────────────────────────────────────
OUT_DIR = DATA_DIR / 'border2_inputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUT_DIR}')
print(f'Bounding box    : {BBOX}')

Loaded user_info.env
Output directory: /home/jovyan/work/digital-twin-group/wildfire-digital-twin/data/border2_inputs
Bounding box    : (-117.1, 32.4, -116.6, 32.8)


## Step 1 — Fire Perimeters

The 6 real WFIGS perimeters are already in the repo.  
We copy them to `border2_inputs/` and print a summary.

In [2]:
DST_PERIMS = OUT_DIR / 'border_2_perimeters.geojson'

if DST_PERIMS.exists():
    print(f'✓ Perimeters already present: {DST_PERIMS.name}')
    gdf = gpd.read_file(DST_PERIMS)

else:
    gdf = None
    source_used = None

    # 1. Firemap (WIFIRE GeoServer WFS) ─────────────────────────────────────
    try:
        print('Trying Firemap (WIFIRE WFS)...')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            gdf_fm = fetch_fire_perimeters('BORDER 2', year=2025, verbose=True)
        # fetch_fire_perimeters returns EPSG:5070 — reproject to WGS84 for storage
        gdf = gdf_fm.to_crs('EPSG:4326')
        source_used = 'Firemap'
        print(f'  ✓ Firemap: {len(gdf)} perimeters')
    except Exception as e:
        print(f'  ✗ Firemap failed: {e}')

    # 2. WFIGS ArcGIS REST ───────────────────────────────────────────────────
    if gdf is None or gdf.empty:
        try:
            print('Trying WFIGS ArcGIS REST...')
            gdf = _fetch_wfigs('BORDER 2', DATE_START, DATE_END, BBOX)
            source_used = 'WFIGS'
            print(f'  ✓ WFIGS: {len(gdf)} perimeters')
        except Exception as e:
            print(f'  ✗ WFIGS failed: {e}')
            gdf = None

    # 3. NIFC year-to-date layer ─────────────────────────────────────────────
    if gdf is None or gdf.empty:
        try:
            print('Trying NIFC YTD layer...')
            gdf = _fetch_nifc('BORDER 2', DATE_START, DATE_END, BBOX)
            source_used = 'NIFC'
            print(f'  ✓ NIFC: {len(gdf)} perimeters')
        except Exception as e:
            print(f'  ✗ NIFC failed: {e}')
            gdf = None

    # 4. Repo fallback ───────────────────────────────────────────────────────
    if gdf is None or gdf.empty:
        fallback = DATA_DIR / 'fire_perimeters' / 'border_2_perimeters.geojson'
        if fallback.exists():
            gdf = gpd.read_file(fallback)
            source_used = f'local fallback ({fallback})'
            print(f'  ✓ Using repo fallback: {fallback.name}')
        else:
            raise FileNotFoundError(
                'All remote sources failed and no local fallback found.\n'
                f'Expected: {fallback}'
            )

    # Save to output dir
    gdf.to_file(DST_PERIMS, driver='GeoJSON')
    print(f'\n✓ Saved {len(gdf)} perimeters → {DST_PERIMS.name}  (source: {source_used})')

# ── Summary ───────────────────────────────────────────────────────────────
print(f'\n{len(gdf)} perimeters  (CRS: {gdf.crs}):')
for i, row in gdf.iterrows():
    # Area: use projected area if already in EPSG:5070, else reproject temporarily
    if gdf.crs and gdf.crs.to_epsg() == 5070:
        area_ac = row.geometry.area / 4046.86
    else:
        geom_proj = gpd.GeoSeries([row.geometry], crs=gdf.crs).to_crs('EPSG:5070').iloc[0]
        area_ac = geom_proj.area / 4046.86
    dt_str = str(row.get('datetime', '—'))
    print(f'  [{i}] {dt_str:<30}  {area_ac:>8.0f} ac')

✓ Perimeters already present: border_2_perimeters.geojson


ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/gdal_env/share/proj failed



6 perimeters  (CRS: EPSG:4326):
  [0] 2025-01-23 00:00:00                  566 ac
  [1] 2025-01-23 00:00:00                  566 ac
  [2] 2025-01-23 00:00:00                  566 ac
  [3] 2025-01-24 00:00:00                 4235 ac
  [4] 2025-01-24 00:00:00                 6258 ac
  [5] 2025-01-26 00:00:00                 6591 ac


## Step 2 — NASA FIRMS Hotspots

Uses the FIRMS CSV API to retrieve Landsat 9 (LANDSAT_NRT) and VIIRS S-NPP
detections for the fire bounding box and date range.  
Requires `FIRMS_MAP_KEY` in `user_info.env`  
(get a free key at https://firms.modaps.eosdis.nasa.gov/api/area/).

In [3]:
FIRMS_CSV = OUT_DIR / 'firms_hotspots.csv'

if FIRMS_CSV.exists():
    df = pd.read_csv(FIRMS_CSV)
    print(f'✓ FIRMS data already present ({len(df)} detections)')
else:
    MAP_KEY = os.getenv('FIRMS_MAP_KEY', '')
    if not MAP_KEY:
        print('[WARN] FIRMS_MAP_KEY not set — using repo fallback')
    else:
        bbox_str = f'{MINX},{MINY},{MAXX},{MAXY}'
        all_rows = []

        # Use archive (non-NRT) products for historical dates
        # Date range is split into ≤10-day chunks (API max per request)
        SENSORS = [
            ('LANDSAT_NRT',    'Landsat 9',   '2025-01-23', '2025-01-27'),
            ('VIIRS_SNPP_NRT', 'VIIRS S-NPP', '2025-01-23', '2025-01-27'),
        ]
        # NRT products cap at 5 days; use standard archive products instead
        SENSORS_ARCHIVE = [
            ('LANDSAT_SP',  'Landsat 9 (archive)',   '2025-01-23', '2025-01-27'),
            ('VIIRS_SNPP_SP', 'VIIRS S-NPP (archive)', '2025-01-23', '2025-01-27'),
        ]

        for product, label, d_start, d_end in SENSORS_ARCHIVE:
            # day_range = number of days from d_start to d_end inclusive
            from datetime import date
            n_days = (date.fromisoformat(d_end) - date.fromisoformat(d_start)).days + 1
            url = (
                f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/'
                f'{MAP_KEY}/{product}/{bbox_str}/{n_days}/{d_start}'
            )
            print(f'  Fetching {label}: {url}')
            try:
                r = requests.get(url, timeout=30)
                r.raise_for_status()
                lines = r.text.strip().splitlines()
                if len(lines) > 1:
                    reader = csv.DictReader(lines)
                    rows = [dict(sensor=product, **row) for row in reader]
                    all_rows.extend(rows)
                    print(f'  {label}: {len(rows)} detections')
                else:
                    print(f'  {label}: no detections in range')
            except Exception as e:
                print(f'  [WARN] {label} fetch failed: {e}')

        if all_rows:
            df = pd.DataFrame(all_rows)
            df.to_csv(FIRMS_CSV, index=False)
            print(f'\n✓ Saved {len(df)} detections → {FIRMS_CSV.name}')
        else:
            print('[INFO] No new detections — using repo fallback')

    # Fallback to existing repo copy regardless
    fallback = DATA_DIR / 'firms_border_2' / 'border_2_firms_hotspots.csv'
    if not FIRMS_CSV.exists() and fallback.exists():
        shutil.copy(fallback, FIRMS_CSV)
        df = pd.read_csv(FIRMS_CSV)
        print(f'✓ Used repo fallback ({len(df)} detections)')

if 'df' in dir() and len(df) > 0:
    print(df[['acq_date','acq_time','satellite','confidence','latitude','longitude']]
          .head(10).to_string(index=False))

✓ FIRMS data already present (334 detections)
  acq_date  acq_time satellite confidence  latitude  longitude
2025-01-23       945         N          n  32.43789 -116.63531
2025-01-23       945         N          n  32.43871 -116.63984
2025-01-23       945         N          n  32.43950 -116.64418
2025-01-23       945         N          n  32.44052 -116.62977
2025-01-23       945         N          n  32.44137 -116.63450
2025-01-23       945         N          h  32.44219 -116.63905
2025-01-23       945         N          n  32.44298 -116.64340
2025-01-23       945         N          n  32.44400 -116.62897
2025-01-23       945         N          h  32.44485 -116.63366
2025-01-23       945         N          n  32.44646 -116.64261


## Step 3 — Weather Observations

Fetches HPWREN / WIFIRE firewx station observations for the fire bounding box
from the Firemap `pylaski/stations` endpoint (same source as `wx_ingestor.py`).  
Covers the full fire interval Jan 23–27 2025 — queries one reading per station
(current state at fetch time, so this is best run as close to the fire dates
as possible; for retrospective runs the Firemap historical endpoint is used).

In [4]:
WX_CSV = OUT_DIR / 'weather_obs.csv'
FIREMAP_WX_URL = 'https://firemap.sdsc.edu/pylaski/stations'
MPS_TO_MPH = 2.23694

if WX_CSV.exists():
    wx_df = pd.read_csv(WX_CSV)
    print(f'✓ Weather data already present ({len(wx_df)} records)')
else:
    params = {
        'camera':    'false',
        'selection': 'boundingBox',
        'obsgroup':  'firewx',
        'minLat':    str(MINY),
        'maxLat':    str(MAXY),
        'minLon':    str(MINX),
        'maxLon':    str(MAXX),
        'callback':  'wxStations',
        '_':         str(int(time.time() * 1000)),
    }
    try:
        resp = requests.get(FIREMAP_WX_URL, params=params, timeout=20)
        resp.raise_for_status()
        text = resp.text.strip()
        if text.startswith('wxStations(') and text.endswith(')'):
            text = text[len('wxStations('):-1]
        data = json.loads(text)
        features = data.get('features', [])
        print(f'  Stations found: {len(features)}')

        records = []
        for feature in features:
            props  = feature.get('properties', {})
            desc   = props.get('description', {})
            coords = feature.get('geometry', {}).get('coordinates', [None, None])
            temp = props.get('temperature', {})
            rh   = props.get('relative_humidity', {})
            ws   = props.get('wind_speed', {})
            wg   = props.get('wind_gust', {})
            wd   = props.get('wind_direction', {})
            if not all([temp.get('value'), ws.get('value'), wd.get('value')]):
                continue
            records.append({
                'station_id':         desc.get('id', 'unknown'),
                'station_name':       desc.get('name', 'unknown'),
                'latitude':           coords[1],
                'longitude':          coords[0],
                'timestamp':          temp.get('timestamp'),
                'temperature_f':      temp.get('value'),
                'relative_humidity':  rh.get('value'),
                'wind_speed_mph':     round(ws['value'] * MPS_TO_MPH, 2),
                'wind_gust_mph':      round(wg['value'] * MPS_TO_MPH, 2) if wg.get('value') else None,
                'wind_direction_deg': wd.get('value'),
                'source':             'firemap-pylaski-stations',
            })

        wx_df = pd.DataFrame(records)
        wx_df.to_csv(WX_CSV, index=False)
        print(f'✓ Saved {len(wx_df)} station records → {WX_CSV.name}')

    except Exception as e:
        print(f'[ERROR] Weather fetch failed: {e}')
        wx_df = pd.DataFrame()

if 'wx_df' in dir() and len(wx_df) > 0:
    print(f'\nNearby weather stations:')
    print(wx_df[['station_name','latitude','longitude',
                 'wind_speed_mph','wind_direction_deg', 
                 'temperature_f', 'relative_humidity']]
          .to_string(index=False))

✓ Weather data already present (34 records)

Nearby weather stations:
                            station_name  latitude  longitude  wind_speed_mph  wind_direction_deg  temperature_f  relative_humidity
                            Border Field  32.54197 -117.09674            6.55               294.6          68.10              77.03
           Tijuana International Airport  32.55000 -116.96667            9.22               250.0          68.00              68.41
                        Otay Mesa Border  32.55817 -116.90048            8.14               311.2          67.46              69.18
San Diego, Brown Field Municipal Airport  32.57528 -116.99306            6.91               280.0          66.20              68.20
                           Otay Mountain  32.59926 -116.84342            7.72               235.9          53.62             100.00
                           OTAY MOUNTAIN  32.60092 -116.84103            8.99               267.0          54.00             100.00
      

## Step 4 — Landscape File (LCP)

Downloads LANDFIRE raster data for the fire bounding box and runs `src/lcpmake`.

Requires in `user_info.env`:
- `LANDFIRE_EMAIL`
- `LANDFIRE_PRIORITY_CODE`

Skip this cell if `landscape.lcp` already exists in the repo root or in `data/`.

In [5]:
LCP_DST = OUT_DIR / 'landscape.lcp'

if LCP_DST.exists():
    print(f'✓ LCP already present: {LCP_DST.name}  ({LCP_DST.stat().st_size/1024**2:.1f} MB)')
else:
    LANDFIRE_EMAIL    = os.getenv('LANDFIRE_EMAIL', '')
    LANDFIRE_PRIORITY = os.getenv('LANDFIRE_PRIORITY_CODE', '')

    if not LANDFIRE_EMAIL or not LANDFIRE_PRIORITY:
        print('[WARN] LANDFIRE_EMAIL or LANDFIRE_PRIORITY_CODE not set in user_info.env')
        print('       Cannot auto-generate LCP — copy landscape.lcp to data/border2_inputs/ manually.')
    else:
        # Build a bounding polygon in WGS84 around the fire center
        pt_gdf = gpd.GeoSeries([Point(FIRE_CENTER_LON, FIRE_CENTER_LAT)], crs='EPSG:4326')
        ignition_poly_wgs84 = (
            pt_gdf.to_crs(pt_gdf.estimate_utm_crs())
                  .buffer(50_000)          # 50 km radius
                  .to_crs('EPSG:4326')
                  .iloc[0]
        )

        rasters = download_landfire_data(
            poly=ignition_poly_wgs84,
            output_dir=OUT_DIR / 'landfire',
            email=LANDFIRE_EMAIL,
            priority_code=LANDFIRE_PRIORITY,
        )
        LCP_DST = generate_lcp_from_rasters(
            output_path=LCP_DST,
            rasters=rasters,
            latitude=FIRE_CENTER_LAT,
        )

# Final check
if LCP_DST.exists():
    print(f'\nLCP ready: {LCP_DST}  ({LCP_DST.stat().st_size/1024**2:.1f} MB)')
else:
    print('\n[ACTION NEEDED] LCP not found — set credentials in user_info.env and re-run.')

✓ LCP already present: landscape.lcp  (110.0 MB)

LCP ready: /home/jovyan/work/digital-twin-group/wildfire-digital-twin/data/border2_inputs/landscape.lcp  (110.0 MB)


## Validation — All Inputs Present?

Run this cell before launching `border2_experiment.ipynb`.

In [6]:
required = {
    'Perimeters GeoJSON':    OUT_DIR / 'border_2_perimeters.geojson',
    'FIRMS hotspots CSV':    OUT_DIR / 'firms_hotspots.csv',
    'Weather stations CSV':  OUT_DIR / 'weather_obs.csv',
    'Landscape LCP':         OUT_DIR / 'landscape.lcp',
}

all_ok = True
print('Input file checklist:')
for label, path in required.items():
    ok = path.exists()
    size = f'{path.stat().st_size/1024:.1f} KB' if ok else '—'
    status = '✓' if ok else '✗ MISSING'
    print(f'  {status}  {label:<26}  {path.name}  ({size})')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('All inputs ready. You can now run border2_experiment.ipynb.')
else:
    print('Some inputs are missing — see warnings above.')

Input file checklist:
  ✓  Perimeters GeoJSON          border_2_perimeters.geojson  (269.6 KB)
  ✓  FIRMS hotspots CSV          firms_hotspots.csv  (30.7 KB)
  ✓  Weather stations CSV        weather_obs.csv  (4.1 KB)
  ✓  Landscape LCP               landscape.lcp  (112673.4 KB)

All inputs ready. You can now run border2_experiment.ipynb.
